# L4M R&D: parameter sets per use case

**Goal:** identify a set of optimised welding parameters for each use case (Fusion welding, Pharma welding, Battery cooling welding).
**Reference flow:** 1 set of optimised parameters for one use case.
**Scenarios per use case:**
- *SoA R&D*: all experiments run physically in the lab with the standard laser.
- *L4M-enabled R&D*: fewer physical experiments, plus digital lab runs, with the joining done by the Lasers4MaaS solution (L4M-solution).

Data come from `databases/L4M_database.bw2package` (`L4M system`).
The background database is ecoinvent 3.12 or BAFU-2026 (set `BACKGROUND`). With BAFU, the ecoinvent inputs of the packages are relinked to BAFU (see `BAFU_LINKS`). See `Diagrams/L4M_RD_parameter_sets.md` for the process flow.
The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [1]:
from pathlib import Path

import bw2data as bd
import numpy as np
import pandas as pd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.constants import (BACKGROUNDS, CT_SCAN_DATABASE, L4M_SOLUTION_TECHNOLOGY, EU27_GB, L4M_DATABASE,
                               METALLURGIC_DATABASE, PACKAGE_DIR, PACKAGE_FILES, PREPOST_PRICES, RD_DATABASE,
                               RD_TESTS_DATABASE, REGIONAL_BASELINE, REGIONAL_ELEC_PRICE)
from l4m_lca.costs import (COST_COLUMNS, LABOUR_HOURS_COLUMN, price_of, regional_labour_costs,
                           system_costs)
from l4m_lca.impacts import BackgroundImpacts, bw2calc_scores, foreground_executions, foreground_flows
from l4m_lca.project import (DEFAULT_INDICATORS, background_inputs, category, check_links, drop_inputs_from,
                             ef31_methods, ensure_package, method_units, open_project, relink_background,
                             select_background, short)
from l4m_lca.rd import (build_rd_database, contribution_rows, make_round, process_label, sankey_links, stack_table,
                        standard_welds_left, validation_test_sets)
from l4m_lca.regional import electricity_markets, system_regional

10:02:56+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


c:\Users\nielenssvan\OneDrive - Universiteit Leiden\Lasers4MaaS - Documenten\General\SystemDesign\3_LCA\l4m_lca\.venv\Lib\site-packages\bw2calc\__init__.py:58: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")


## Parameters

In [2]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BG = select_background("ecoinvent")
BACKGROUND = BG.name
ECOINVENT_DATABASE = BACKGROUNDS["ecoinvent"]["database"]   # the packages link to this database
REIMPORT_L4M = False   # True: delete and import L4M system again (discards edits made in the project)

INDICATORS = DEFAULT_INDICATORS

# BAFU counterparts of the ecoinvent inputs of the packages: (ecoinvent name, location) -> (BAFU name, location).
# Used only with BACKGROUND = "bafu". BAFU has ecoinvent v2 names.
BAFU_LINKS = {
    ("market group for electricity, medium voltage", "RER"): ("Electricity, medium voltage, at grid", "RER"),
    ("market for electricity, medium voltage", "GB"):        ("Electricity, medium voltage, at grid", "GB"),
    ("market for argon, liquid", "RER"):                     ("Argon, liquid, at plant", "RER"),
    ("market for aluminium alloy, AlMg3", "GLO"):            ("Aluminium alloy, AlMg3, at plant", "RER"),
    # ecoinvent cut-off scrap has no burden; the BAFU scrap carries its transport
    ("aluminium scrap, new, Recycled Content cut-off", "GLO"): ("Aluminium scrap, new, at plant", "RER"),
    ("market for isopropanol", "RER"):                       ("Isopropanol, at plant", "RER"),
    ("market for tissue paper", "GLO"):                      ("Kraft paper, bleached, at plant", "RER"),  # PROXY: BAFU has no tissue paper
    ("market for tap water", "Europe without Switzerland"):  ("Tap water, at user", "RER"),
    ("market for chemical, organic, unspecified", "GLO"):    ("Chemicals organic, at plant", "GLO"),
    ("market for paper, woodfree, uncoated", "RER"):         ("Paper, woodfree, uncoated, at regional storage", "RER"),
    ("market for silicon carbide", "GLO"):                   ("Silicon carbide, at plant", "RER"),
    ("market for phenolic resin", "RER"):                    ("Phenolic resin, at plant", "RER"),
    ("market for polypropylene, granulate", "RER"):          ("Polypropylene, granulate, at plant", "RER"),
    ("injection moulding", "RER"):                           ("Injection moulding", "RER"),
}

L4M_SOLUTION_ENERGY_FACTOR = 0.9   # The L4M-solution uses 10% less electricity than the standard laser (all electricity of the joining steps), as the joining model
OPTIMIZATION_KWH = 0.01   # one Bayesian optimization run on a workstation: about 5 min at 120 W (estimate; range 0.002-0.2 kWh). None: use one "Prediction" process
INCLUDE_MATERIAL = False  # True adds the material inputs (Raw material sourcing: aluminium of the coupons)
# CT scan of the joined welds: scanner power x scan time. The CT-scan package gave 3 kWh per scan;
# 3 kW is taken as the power. Scan time is an estimate, to be replaced by partner data.
CT_SCAN_KW = 3.0
CT_SCAN_H = 0.5          # estimate: 30 min per scan
CT_SCAN_KWH = CT_SCAN_KW * CT_SCAN_H

# Validation tests of the prototype (D13.1, rd_tests.ipynb): test process code -> tests per set of tests.
# One set of tests per PROTOTYPES_PER_TEST_SET prototypes (rounded up). Fusion welding has no D13.1 tests.
PROTOTYPES_PER_TEST_SET = 10
VALIDATION_TESTS = {
    "Battery cooling welding": {   # battery cooling plate
        "leak_tightness_test": 1, "macrography_seam_analysis": 1, "burst_pressure_test": 1,
        "pressure_cycle_test": 1, "thermal_shock_test": 1},
    "Pharma welding": {            # heat exchanger prototype, AISI 316L (freeze dryer shelf)
        "hx_cleaning": 1, "hx_resistance_spot_welding": 1, "hx_pickling_electropolishing": 1,
        "hx_quality_test": 1},
    "Fusion welding": {},
}

# Partner lab data (lab data collection): per round, the number of coupon welds, prototype welds,
# digital lab runs, optimization runs, verifications (CT scans of the joined welds) and sets of validation tests.
# "joining" is the technology of the coupon and prototype joining.
# The digital lab reduces the prototype welds in the same proportion as the coupon welds.
COUPONS_SOA = (125, 125)    # coupon welds per round, SoA R&D
COUPONS_L4M = (75, 75)    # coupon welds per round, L4M-enabled R&D
PROTOTYPES_SOA = 40
PROTOTYPES_L4M = 20

def scenarios(prototype_soa, prototype_l4m):
    return {
        "SoA R&D": {"joining": "Standard laser", "rounds": {
            "Coupon round 1":    make_round(coupon=COUPONS_SOA[0]),
            "Coupon round 2":    make_round(coupon=COUPONS_SOA[1]),
            "Prototype round 1": make_round(prototype=prototype_soa,
                                             tests=validation_test_sets(prototype_soa, PROTOTYPES_PER_TEST_SET))}},
        "L4M-enabled R&D": {"joining": L4M_SOLUTION_TECHNOLOGY, "rounds": {
            "Coupon round 1":    make_round(coupon=COUPONS_L4M[0], digital=1),
            "Coupon round 2":    make_round(coupon=COUPONS_L4M[1], optimization=4),
            "Prototype round 1": make_round(prototype=prototype_l4m,
                                             tests=validation_test_sets(prototype_l4m, PROTOTYPES_PER_TEST_SET))}},
    }

USE_CASE_NAMES = ["Fusion welding", "Pharma welding", "Battery cooling welding"]
RD_SCENARIOS = {uc: scenarios(prototype_soa=PROTOTYPES_SOA, prototype_l4m=PROTOTYPES_L4M) for uc in USE_CASE_NAMES}

# --- Operating costs (same cost data as pre_post.ipynb)
# Prices (EUR per unit of the background dataset). A key matches every background input whose lower-case name
# contains it (ecoinvent and BAFU names); the first match wins. Electricity: ELECTRICITY_PRICES by location.
P = PREPOST_PRICES
RD_PRICES = {
    "tap water":           P["water"],               # pre_post.ipynb
    "sodium hydroxide":    P["sodium_hydroxide"],    # pre_post.ipynb
    "nitric acid":         P["nitric_acid"],         # pre_post.ipynb
    "hydrochloric acid":   P["hydrochloric_acid"],   # pre_post.ipynb
    "silicon carbide":     P["grinding_wheel"],      # pre_post.ipynb (SiC grinding paper)
    "aluminium oxide":     P["abrasive"],            # pre_post.ipynb
    "lubricating oil":     P["coolant"],             # pre_post.ipynb
    "tungsten carbide":    P["tool_material"],       # pre_post.ipynb
    "argon":               5.87,   # EUR/kg, joining_task.ipynb (10.47 EUR/m3 / 1.784 kg/m3)
    # PLACEHOLDER prices (EUR/kg): not in pre_post.ipynb. Replace with supplier quotes.
    "aluminium scrap":     0.0,    # cut-off scrap of the coupons: no price
    "aluminium alloy":     3.10,    # AlMg3 sheet (coupons; only with INCLUDE_MATERIAL = True)
    "isopropanol":         1.5,
    "tissue paper":        6.0,    # 42.83 GBP = 51.15 EUR, 8.538 kg
    "kraft paper":         6.0,    # BAFU proxy of tissue paper
    "paper, woodfree":     1.0,
    "chemical, organic":   2.0,
    "chemicals organic":   2.0,    # BAFU name
    "phenolic resin":      2.5,    # hot-mounting resin
    "polypropylene":       1.2,
    "injection moulding":  1.0,    # per kg of moulded part
    "hydrogen fluoride":   2.0,    # Ecoinvent prices data
    "phosphoric acid":     1.0,
    "sulfuric acid":       0.1,
    "sulphuric acid":      0.1,    # BAFU name
    "acetone":             0.5,    # UN Comtrade
    "ethylene glycol":     1.0,
    "copper":              11.7,   # TradingEconomic.com/commodity/copper
}
ELECTRICITY_PRICES = REGIONAL_ELEC_PRICE   # EUR/kWh by location: RER 0.164 (Eurostat), GB 0.3151 (DESNZ)
LABOUR_COST = regional_labour_costs()[REGIONAL_BASELINE]   # 35.0 EUR/h, EU27 average, as pre_post.ipynb

# Operator hours per unit of the reference product of each foreground process (by process name).
# 1 operator works during the whole process time. Processes without an entry count 0 h; the cost cell lists
# the ones that use background inputs directly. Rounds, systems and sets of tests are only containers.
# Copies for the L4M-solution keep the name of the process, so they get the same hours.
PROCESS_HOURS = {
    # Coupons (PLACEHOLDER estimates)
    "01_Cutting":                 0.05,   # per cut coupon (3 min)
    "02_Cleaning":                0.05,   # per clean coupon (3 min)
    "001_Physical_Laser welding": 0.25,   # per weld coupon: clamping, welding, unloading (15 min)
    # Prototype welding, per prototype (PLACEHOLDER estimates: set-up, clamping, welding)
    "Fusion welding":             1.0,
    "Pharma welding":             1.0,
    "Battery cooling welding":    1.0,
    # Verification and digital lab
    "CT-scan X-ray":              CT_SCAN_H,   # scan time (estimate above)
    "Optimization":               5 / 60,      # one optimization run (about 5 min, see OPTIMIZATION_KWH)
    "Platform":                   0.5,         # PLACEHOLDER: set-up of the digital lab per round
    "Prediction":                 0.5,         # PLACEHOLDER: prediction runs per round
    "Raw material sourcing":      0.0,         # purchase only
    # Validation tests (rd_tests.ipynb), per test: D13.1 durations and the midpoints of its ESTIMATES
    "Leak tightness test, battery cooling plate":              99 / 3600,           # D13.1: 99 s cycle
    "Macrography seam analysis, battery cooling plate":        2.0,                 # PLACEHOLDER: 4 cross-sections
    "Burst pressure test, battery cooling plate":              (200 + 600) / 3600 + 0.15,  # 20 bar at 0.1 bar/s, 2 x 5 min holds, set-up
    "Pressure cycle (fatigue) test, battery cooling plate":    120_000 / 3600,      # D13.1: 120,000 cycles at 1 Hz (33.3 h)
    "Thermal shock test, battery cooling plate":               10 * 10 / 60,        # D13.1: 10 cycles of 5 + 5 min
    "Cleaning, heat exchanger prototype":                      0.25,                # estimate: 10-20 min
    "Resistance spot welding, heat exchanger prototype":       0.375,               # estimate: 15-30 min
    "Pickling and electropolishing, heat exchanger prototype": 0.165 + 0.175,       # estimate: polishing + bath heating
    "Quality test (leak test), heat exchanger prototype":      0.75,                # estimate: 30-60 min
}

In [3]:
# Saved processes: label -> (database, code). The comment gives the reference flow of each process:
# the amount of its reference product that it produces. All its input amounts refer to this flow.
L4M_PROCESSES = {
    "001_Physical lab (battery cooling pre-processing + welding)": "a7cf0ea95c2a499aa8cf9eb0ba72142c",  # 1 "1 set of optimised parameters"
    "001_Physical_Cutting":       "97a5ed8baec4406f879d5cff60e87345_copy1",  # 500 "Cut coupon"
    "001_Physical_Laser welding": "5a81aade45924c56854be233364a0435_copy1",  # 250 "Weld coupon"
    "001_Pyshical_Cleaning":      "fbbf6041c24c4e28b6c23a3ab0eebf0a_copy1",  # 500 "Clean coupon"
    "Battery cooling welding":    "68f4a5e21189435ba9b0a376e068edbc",        # 1 "Battery cooling plate"
    "CT-scan X-ray":              "93d9467878cb43ba9798a256e98512f5_copy1",  # 1 "Tested coupon" (location GB); only its electricity input is used
    "Digital lab processes":      "d273f34fee714ae0b25dbfb39576ab5b",        # 1 "Digital lab processes"
    "Fusion welding":             "68f4a5e21189435ba9b0a376e068edbc_copy2",  # 1 "Pipe welding"
    "Pharma welding":             "68f4a5e21189435ba9b0a376e068edbc_copy1",  # 1 "Freeze dryer shelf"
    "Platform":                   "50a84844a3de4677926d511e171e31d5",        # 1 "Platform"
    "Prediction":                 "8fbb7937f5224de9b836ae54858b6fe1",        # 1 "Prediction"
}
METALLURGIC_PROCESSES = {
    "01_Cutting":       "97a5ed8baec4406f879d5cff60e87345",  # 132 "Cut coupon"
    "02_Cleaning":      "fbbf6041c24c4e28b6c23a3ab0eebf0a",  # 1 "Clean coupon"
    "03_Laser welding": "5a81aade45924c56854be233364a0435",  # 1 "Weld coupon"
    "04_Cutting":       "7faf722864b94ed797cdd406d22120db",  # 3 "Weld cross-sections"
    "05_Hot-mounting":  "06923c5d5a4d416d87c1702173dc648a",  # 1 "Mounted sample"
    "06_Grinding":      "f00138e49c5843caa7a1e7d8f33f5986",  # 6 "Grinded sample"
    "07_Polishing":     "53c6381979c942e79ed1f043a8b184ed",  # 6 "Polished sample"
    "08_Etching":       "92039648c9a64364ab22f228c0f4b44d",  # 18 "Etched sample"
    "09_Test":          "32bb94122a4542ad8af12fb8ea34a26e",  # 18 "Tested sample"
}

# Plot groups: process type -> {process label: colour} (legend order)
PROCESS_TYPES = {
    "Pre-processing":  {"Raw material sourcing": "#0f3f7a", "Cutting": "#2a78d6", "Cleaning": "#8fb8ee"},
    "Joining":         {"Coupon joining": "#eb6834", "Prototype joining": "#f4a582"},
    "Testing":         {"CT-scan X-ray": "#1baf7a",
                        "Leak tightness test, battery cooling plate": "#0b6e4f",
                        "Macrography seam analysis, battery cooling plate": "#3fbf8f",
                        "Burst pressure test, battery cooling plate": "#7fd8b2",
                        "Pressure cycle (fatigue) test, battery cooling plate": "#145a3a",
                        "Thermal shock test, battery cooling plate": "#b9ead3",
                        "Quality test (leak test), heat exchanger prototype": "#2e8b57"},
    "Prototype manufacturing": {"Cleaning, heat exchanger prototype": "#6b4f2a",
                        "Resistance spot welding, heat exchanger prototype": "#a07a45",
                        "Pickling and electropolishing, heat exchanger prototype": "#d2b48c"},
    "Digital lab":     {"Platform": "#eda100", "Prediction": "#e87ba4", "Optimization": "#f6cf6b"},
}

# Regional analysis: EU27 plus GB (the physical lab is modelled with GB electricity). Disabled by default.
SHOW_SANKEY = False
SHOW_REGIONAL = False
REGIONAL_COUNTRIES = EU27_GB

## Project and imported databases

In [5]:
background = open_project(BG.project, BG.database)
print(f"Background: {BG.database} (project {BG.project})")
# L4M system links to the metallurgic and CT-scan packages, so they come first. The CT-scan package is
# needed only for that import: its chain (cutting, cleaning and a 38.7 kWh coupon weld) is then removed
# from "CT-scan X-ray", and the package is deleted. The R&D model uses the CT scan alone (CT_SCAN_KWH).
if REIMPORT_L4M or L4M_DATABASE not in bd.databases or len(bd.Database(L4M_DATABASE)) == 0:
    ensure_package(CT_SCAN_DATABASE, Path(PACKAGE_DIR) / PACKAGE_FILES[CT_SCAN_DATABASE])
ensure_package(METALLURGIC_DATABASE, Path(PACKAGE_DIR) / PACKAGE_FILES[METALLURGIC_DATABASE])
ensure_package(L4M_DATABASE, Path(PACKAGE_DIR) / PACKAGE_FILES[L4M_DATABASE], force=REIMPORT_L4M)
print(f"Removed {drop_inputs_from(L4M_DATABASE, CT_SCAN_DATABASE)} links from {L4M_DATABASE} to {CT_SCAN_DATABASE}.")
if CT_SCAN_DATABASE in bd.databases:
    del bd.databases[CT_SCAN_DATABASE]
    print(f"Deleted {CT_SCAN_DATABASE} from the project.")
PACKAGES = [L4M_DATABASE, METALLURGIC_DATABASE]
if BG.database != ECOINVENT_DATABASE:
    # The packages link to ecoinvent: point those inputs to the other background
    relinked = relink_background(PACKAGES, ECOINVENT_DATABASE, BG.database, BAFU_LINKS)
    print(f"Relinked {sum(relinked.values())} exchanges from {ECOINVENT_DATABASE} to {BG.database}.")
check_links(PACKAGES)
used = background_inputs(PACKAGES, set(PACKAGES))
assert used == {BG.database}, f"The packages take inputs from {used}, not only from {BG.database}"

methods = ef31_methods(INDICATORS)
units = method_units(methods)
indicators = [category(m) for m in methods]
DEFAULT_INDICATOR = next((i for i in indicators if i.startswith("climate change")), indicators[0])

Background: ecoinvent-3.12-cutoff (project Lasers4MaaS)


OSError: [Errno 22] Invalid argument

In [6]:
l4m = {label: bd.Database(L4M_DATABASE).get(code) for label, code in L4M_PROCESSES.items()}
metallurgic = {label: bd.Database(METALLURGIC_DATABASE).get(code) for label, code in METALLURGIC_PROCESSES.items()}

# Reference flow of each saved process, to check the comments above
pd.DataFrame(
    [{"Database": a["database"], "Process": label, "Reference product": a.get("reference product"),
      "Location": a.get("location"), "Unit": a["unit"],
      "Amount": next(e["amount"] for e in a.exchanges() if e["type"] == "production")}
     for procs in (l4m, metallurgic) for label, a in procs.items()])

UnknownObject: 

## R&D product systems

One product system per use case and scenario. The set of parameters is identified in rounds (coupon round 1, coupon round 2, prototype round 1).
In the L4M-enabled scenario, the digital platform investigates parameter sets, and an optimization run uses the results of the successful tests before the next physical round.
Every physical round is verified by a CT scan of the joined welds (scanner power x scan time).
Each prototype round also has one set of D13.1 validation tests (`VALIDATION_TESTS`) per 10 prototypes (`PROTOTYPES_PER_TEST_SET`, rounded up), from the database `RD_validation_tests` of `rd_tests.ipynb`: the battery cooling plate tests for Battery cooling welding, and the heat exchanger chain for Pharma welding.
With `INCLUDE_MATERIAL = False`, the material inputs (aluminium of the coupons) are removed.

In [5]:
use_cases = {uc: l4m[uc] for uc in USE_CASE_NAMES}
if OPTIMIZATION_KWH is None:   # electricity of one "Prediction" process, per unit of its reference flow
    pred = l4m["Prediction"]
    pred_out = next(e["amount"] for e in pred.exchanges() if e["type"] == "production")
    OPTIMIZATION_KWH = sum(e["amount"] for e in pred.exchanges()
                           if e["type"] == "technosphere" and "electricity" in e.input["name"].lower()) / pred_out
    print(f"Electricity of one optimization run = one Prediction process: {OPTIMIZATION_KWH:.4g} kWh")
assert RD_TESTS_DATABASE in bd.databases, f"Run rd_tests.ipynb with BACKGROUND = {BACKGROUND!r} first: it writes {RD_TESTS_DATABASE}"
validation_tests = {uc: {(RD_TESTS_DATABASE, code): n for code, n in tests.items()} for uc, tests in VALIDATION_TESTS.items()}
for uc, tests in validation_tests.items():
    for key in tests:
        bd.get_activity(key)   # raises if rd_tests.ipynb did not write this test
systems = build_rd_database(RD_DATABASE, BG.database, l4m, metallurgic, use_cases,
                            RD_SCENARIOS, L4M_SOLUTION_ENERGY_FACTOR, OPTIMIZATION_KWH, CT_SCAN_KWH,
                            validation_tests=validation_tests, include_material=INCLUDE_MATERIAL)
list(systems)

NameError: name 'l4m' is not defined

## Impact calculation

Each product system is walked through the foreground processes to the background inputs (background datasets, biosphere flows).
The impact of 1 unit of each background input is calculated one time, then scaled by the amount for 1 set of parameters.
The next cell checks the totals against a regular `bw2calc` LCA.

In [ ]:
FOREGROUND = {L4M_DATABASE, METALLURGIC_DATABASE, RD_TESTS_DATABASE, RD_DATABASE}
impacts = BackgroundImpacts(methods, next(iter(background)))
flows = {k: foreground_flows(act, FOREGROUND) for k, act in systems.items()}
contribs = {k: impacts.by_flow(f) for k, f in flows.items()}
totals = {k: sum(v for _, v in c) for k, c in contribs.items()}

for (uc, sc), f in flows.items():
    if RD_SCENARIOS[uc][sc]["joining"] == L4M_SOLUTION_TECHNOLOGY:
        left = standard_welds_left(f, use_cases)
        assert not left, f"Standard-laser weld left in {uc} | {sc}: {left}"
for k, act in systems.items():
    assert np.allclose(totals[k], bw2calc_scores(act, methods), rtol=1e-6), f"Walk does not reproduce bw2calc for {k}"
print(f"Walk matches bw2calc for all {len(systems)} product systems.")

pd.DataFrame({k: dict(zip(indicators, v)) for k, v in totals.items()}).T.rename_axis(["Use case", "Scenario"])

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)
/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


Walk matches bw2calc for all 6 product systems.


acidification  \
Use case                Scenario                         
Fusion welding          SoA R&D               0.049744   
                        L4M-enabled R&D       0.038145   
Pharma welding          SoA R&D               0.173058   
                        L4M-enabled R&D       0.100536   
Battery cooling welding SoA R&D               1.370673   
                        L4M-enabled R&D       0.699695   

                                         eutrophication: terrestrial  \
Use case                Scenario                                       
Fusion welding          SoA R&D                             0.085876   
                        L4M-enabled R&D                     0.065834   
Pharma welding          SoA R&D                             0.242936   
                        L4M-enabled R&D                     0.145486   
Battery cooling welding SoA R&D                             2.174358   
                        L4M-enabled R&D                     1.111734   

                                         human toxicity: carcinogenic  \
Use case                Scenario                                        
Fusion welding          SoA R&D                          2.100961e-09   
                        L4M-enabled R&D                  1.562443e-09   
Pharma welding          SoA R&D                          7.026292e-09   
                        L4M-enabled R&D                  4.050141e-09   
Battery cooling welding SoA R&D                          5.987081e-08   
                        L4M-enabled R&D                  3.048439e-08   

                                         human toxicity: carcinogenic, inorganics  \
Use case                Scenario                                                    
Fusion welding          SoA R&D                                      9.969887e-10   
                        L4M-enabled R&D                              7.715283e-10   
Pharma welding          SoA R&D                                      3.928394e-09   
                        L4M-enabled R&D                              2.253060e-09   
Battery cooling welding SoA R&D                                      3.043194e-08   
                        L4M-enabled R&D                              1.551242e-08   

                                         human toxicity: carcinogenic, organics  \
Use case                Scenario                                                  
Fusion welding          SoA R&D                                    1.103972e-09   
                        L4M-enabled R&D                            7.909148e-10   
Pharma welding          SoA R&D                                    3.097898e-09   
                        L4M-enabled R&D                            1.797081e-09   
Battery cooling welding SoA R&D                                    2.943887e-08   
                        L4M-enabled R&D                            1.497198e-08   

                                         particulate matter formation  \
Use case                Scenario                                        
Fusion welding          SoA R&D                          3.307114e-07   
                        L4M-enabled R&D                  2.303376e-07   
Pharma welding          SoA R&D                          8.819765e-07   
                        L4M-enabled R&D                  5.085049e-07   
Battery cooling welding SoA R&D                          5.247866e-06   
                        L4M-enabled R&D                  2.692664e-06   

                                         photochemical oxidant formation: human health  \
Use case                Scenario                                                         
Fusion welding          SoA R&D                                               0.026205   
                        L4M-enabled R&D                                       0.020120   
Pharma welding          SoA R&D                                               0.081740   
                        L4M-enabl

## Operating costs

Costs of 1 set of optimised parameters, per product system:
- **Consumable costs**: price x amount of every background input of the system (all foreground levels), with `RD_PRICES` and `ELECTRICITY_PRICES` (the cost data of `pre_post.ipynb`, plus placeholders for the other inputs).
- **Personnel costs**: 1 operator during the time of every process (`PROCESS_HOURS` x number of units) x `LABOUR_COST` (EU27 average).
- **Nominal OPEX**: consumable + personnel costs.

The cell lists the inputs without a price and the processes without a time: they count 0.

In [4]:
FOREGROUND = {L4M_DATABASE, METALLURGIC_DATABASE, RD_TESTS_DATABASE, RD_DATABASE}
executions = {k: foreground_executions(act, FOREGROUND) for k, act in systems.items()}
costs, unpriced, untimed = {}, set(), set()
for k in systems:
    costs[k], no_price, no_time = system_costs(flows[k], executions[k], RD_PRICES, ELECTRICITY_PRICES,
                                               PROCESS_HOURS, LABOUR_COST)
    unpriced |= no_price
    untimed |= no_time
if unpriced:
    print("No price (cost 0):", sorted(unpriced))
if untimed:
    print("No process time (0 h):", sorted(untimed))

cost_df = pd.DataFrame(costs).T.rename_axis(["Use case", "Scenario"])[[LABOUR_HOURS_COLUMN, *COST_COLUMNS]]
cost_df.to_csv(f"{BG.results}/L4M_RD_costs.csv")
cost_df.round(2)

NameError: name 'systems' is not defined

In [ ]:
# Consumable costs (EUR) per process: the direct background inputs of each process, by plot label
weld_names = {p["name"] for p in use_cases.values()}
breakdown = {}
for k, fl in flows.items():
    for f in fl:
        price = price_of(f.node, RD_PRICES, ELECTRICITY_PRICES) if f.kind == "technosphere" else None
        if price:
            label = process_label(bd.get_activity(f.path[-1])["name"], weld_names)
            breakdown.setdefault(k, {}).setdefault(label, 0.0)
            breakdown[k][label] += f.amount * price
pd.DataFrame(breakdown).T.rename_axis(["Use case", "Scenario"]).fillna(0.0).round(2)

## Stacked bar: SoA R&D vs L4M-enabled R&D

Impact of 1 set of optimised parameters, split by the process that causes it directly. The legend groups the processes into pre-processing, joining, testing, prototype manufacturing and digital lab.

In [ ]:
weld_names = {p["name"] for p in use_cases.values()}
rows = contribution_rows(contribs, weld_names, indicators)   # one row per system and background flow

display(plots.grouped_stacked_bars(lambda ind: stack_table(rows, ind), indicators, DEFAULT_INDICATOR, units,
                                   PROCESS_TYPES,
                                   title=lambda ind: f"{short(ind)} per set of optimised parameters"))

## Sankey: where the impact flows

Background inputs (left) feed the foreground processes, which pass their cumulative impact on to the set of parameters (right). Set `SHOW_SANKEY = True` to show it.

In [ ]:
if SHOW_SANKEY:
    db_tag = {L4M_DATABASE: "L4M", METALLURGIC_DATABASE: "Metallurgic", RD_TESTS_DATABASE: "Tests", RD_DATABASE: ""}

    def node_label(key):
        act = bd.get_activity(key)
        tag = db_tag.get(act["database"], "")
        return f"{act['name']} ({tag})" if tag else act["name"]

    display(plots.sankey(
        links_of=lambda k, ind: sankey_links(contribs[k], indicators.index(ind), node_label),
        systems={f"{uc} | {sc}": (uc, sc) for uc, sc in systems},
        indicators=indicators, default=DEFAULT_INDICATOR, units=units,
        totals=lambda k, ind: totals[k][indicators.index(ind)],
        root_name=lambda k: systems[k]["name"]))

## Regional variation: EU electricity markets

Every medium-voltage electricity input (RER and the GB electricity of the physical lab) is replaced by the market of one country. Left: SoA R&D, right: L4M-enabled R&D. Set `SHOW_REGIONAL = True` to show it.

In [ ]:
if SHOW_REGIONAL:
    markets = electricity_markets(background, REGIONAL_COUNTRIES, BG.country_electricity)
    reg = system_regional(systems, flows, impacts, markets, baseline="As modelled")
    display(plots.paired_maps(
        value=lambda uc, sc, c, ind: reg[(uc, sc), c][indicators.index(ind)],
        use_cases=list(RD_SCENARIOS), scenarios=["SoA R&D", "L4M-enabled R&D"],
        indicators=indicators, default=DEFAULT_INDICATOR, units=units, countries=REGIONAL_COUNTRIES,
        baseline="As modelled", title=lambda uc, ind: f"{uc}: {short(ind)} per set of parameters"))

In [ ]:
# Export for the report: GHG and electricity per process, and the main background inputs (GHG)
report = pd.DataFrame(rows).rename(columns={DEFAULT_INDICATOR: "ghg_kg_co2_eq"})
rd_df = report.groupby(["use_case", "scenario", "process"], sort=False, as_index=False)[
    ["ghg_kg_co2_eq", "electricity_kwh"]].sum()
rd_df.to_csv(f"{BG.results}/L4M_RD_results.csv", index=False)
top_df = (report.groupby(["use_case", "scenario", "process", "background_input", "unit"], sort=False,
                         as_index=False)[["amount", "ghg_kg_co2_eq"]].sum()
          .sort_values("ghg_kg_co2_eq", ascending=False))
top_df.to_csv(f"{BG.results}/L4M_RD_top_inputs.csv", index=False)
print(f"Saved {BG.results}/L4M_RD_results.csv and L4M_RD_top_inputs.csv")
rd_df.pivot_table(index=["use_case", "scenario"], values=["ghg_kg_co2_eq", "electricity_kwh"], aggfunc="sum", sort=False)

Saved results/L4M_RD_results.csv and L4M_RD_top_inputs.csv


ghg_kg_co2_eq  electricity_kwh
use_case                scenario                                       
Fusion welding          SoA R&D               9.197154        18.846307
                        L4M-enabled R&D       7.522510        17.592222
Pharma welding          SoA R&D              28.879816        64.087248
                        L4M-enabled R&D      17.520585        40.647914
Battery cooling welding SoA R&D             305.108611       749.060768
                        L4M-enabled R&D     155.710074       383.343174